### **Notebook 3 (etapa 3 y 4): Entrenamiento y evaluación de regresión XGBoost**

In [ ]:
# pip install xgboost joblib
import pandas as pd  # Permite el manejo y análisis de estructuras de datos (DataFrames)
import numpy as np  # Facilita la realización de cálculos numéricos y el manejo de matrices
import os  # Interacción con el sistema operativo (creación y verificación de rutas/directorios)
import gc  # Recolección de basura (Garbage Collector) para liberar memoria RAM
import time  # Medición de los tiempos de ejecución de las tareas
import joblib  # Serialización y guardado de los modelos entrenados de Machine Learning en disco
from sklearn.model_selection import GridSearchCV  # Realiza búsquedas exhaustivas de hiperparámetros
from sklearn.model_selection import StratifiedKFold  # Divide los datos en pliegues preservando la proporción de cada clase
from sklearn.base import clone  # Permite clonar estimadores sin copiar los datos originales
import xgboost as xgb  # Algoritmo principal de ensamble basado en Gradient Boosting para clasificación
from sklearn.preprocessing import label_binarize  # Convierte etiquetas multiclase en un formato binario (One-vs-Rest)
from sklearn.metrics import (  # Colección de funciones para evaluar el rendimiento del modelo
    f1_score,
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    classification_report
)

def entrenar_evaluar_xgb(target_name):
    """
    Descripción:
        Entrena, regulariza, optimiza y evalúa un modelo XGBoost (Extreme Gradient Boosting).
        Se encarga de balancear los datos, buscar la mejor combinación de hiperparámetros que 
        ofrezca estabilidad, evaluar el conjunto de pruebas extrayendo métricas y Feature Importances,
        y exportar todos los resultados y el modelo óptimo a disco (preparado para SHAP).

    Entradas:
        - target_name (str): Nombre exacto de la columna que representa la variable objetivo (target) a predecir.

    Salidas:
        - None: La función no retorna elementos directamente en memoria, pero guarda en disco local:
            1. Un archivo CSV con los resultados de la validación cruzada (GridSearch).
            2. El modelo serializado (.pkl) con la mejor configuración de hiperparámetros.
            3. Un archivo CSV con el reporte de métricas (no explícito en exportación directa de dict en esta versión, pero imprime en consola).
            4. Un archivo CSV con la importancia relativa de las variables (Feature Importances).
    """
    # 1. Configuración inicial
    # Definir el directorio de lectura de datos
    dir_datos = "../../Datos/Datasets Finales"
    # Definir y crear el directorio para almacenar los resultados si no existe
    dir_resultados = "../../Resultados/Resultados (etapa 3 y 4)/XGBoost"
    os.makedirs(dir_resultados, exist_ok=True)

    # Lista de variables que no deben ser incluidas como features predictoras en el modelo
    cols_excluir = ['CONSUMO_RECURSOS', 'SEVERIDAD', 'MORTALIDAD', 'CATEGORIA_CANCER'] 

    print("="*60)
    print(f"Iniciando entrenamiento y evaluación de XGBOOST para la variable objetivo: {target_name.upper()}")
    print("="*60) 

    # [1/5] y [2/5] Carga y Balanceo de datos
    print("[1/5] Cargando datasets de entrenamiento...")
    # Lectura del dataset con casos positivos/oncológicos
    df_onco_train = pd.read_csv(os.path.join(dir_datos, "dataset_entrenamiento_onco.csv"), low_memory=False)
    # Lectura del dataset con casos negativos/de control
    df_control_train = pd.read_csv(os.path.join(dir_datos, "dataset_entrenamiento_control.csv"), low_memory=False)

    print("[2/5] Generando muestra balanceada...")
    # Obtener el número de registros oncológicos para equilibrar las clases
    n_onco = len(df_onco_train)
    # Unificar los subconjuntos tomando una muestra aleatoria de controles equivalente al tamaño oncológico
    df_train_maestro = pd.concat([df_onco_train, df_control_train.sample(n=n_onco, random_state=42)], ignore_index=True)
    # Eliminar dataframes intermedios y llamar al colector de basura para liberar memoria RAM
    del df_onco_train, df_control_train; gc.collect()

    # Filtrar las columnas para dejar únicamente las características (features)
    features = [col for col in df_train_maestro.columns if col not in cols_excluir]
    # Asignar features a X_train y target a y_train
    X_train = df_train_maestro[features]
    y_train = df_train_maestro[target_name]
    
    # Identificar la cantidad de clases presentes para saber si es clasificación binaria o multiclase
    clases_unicas = np.unique(y_train)
    is_multiclass = len(clases_unicas) > 2

    print(f"      -> Dimensiones entrenamiento: {X_train.shape} | Clases: {len(clases_unicas)}")

    # [3/5] Configurar Grid Search Regularizado
    print("[3/5] Configurando Grid Search CV (K=5)...")
    # Establecer la validación cruzada estratificada para preservar la proporción de clases en 5 cortes
    cv_estrategia = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    # Configurar el estimador base dependiendo de si es un problema binario o multiclase
    if not is_multiclass:
        # Calcular el peso de la clase positiva para balanceo interno en XGBoost binario
        conteo_clases = y_train.value_counts()
        peso_clase_positiva = conteo_clases[0] / conteo_clases[1]
        xgb_base = xgb.XGBClassifier(
            tree_method='hist', scale_pos_weight=peso_clase_positiva,
            random_state=42, n_jobs=-1
        )
    else:
        # Instanciar el modelo base para problemas multiclase
        xgb_base = xgb.XGBClassifier(tree_method='hist', random_state=42, n_jobs=-1)

    # Definir la grilla de hiperparámetros a explorar
    espacio_hiperparametros = {
        'learning_rate': [0.01, 0.1, 0.3], # Tasa de aprendizaje o magnitud de corrección de errores por árbol
        'max_depth': [3, 6, 10] # Profundidad máxima permitida para cada árbol individual
    }

    # Configurar el buscador exhaustivo con validación cruzada optimizando el 'f1_macro'
    grid_search = GridSearchCV( 
        estimator=xgb_base, # Usa el estimador base de XGBoost creado arriba
        param_grid=espacio_hiperparametros, # Inyecta las configuraciones posibles de hiperparámetros
        cv=cv_estrategia, # Asigna los 5 pliegues asegurados mediante StratifiedKFold
        scoring='f1_macro', # Establece F1-Macro como métrica principal a maximizar
        n_jobs=1, # Evalúa los pliegues secuencialmente para no agotar la memoria RAM
        verbose=3 # Activa el reporte detallado por iteración en la consola
    )

    # [4/5] Entrenar y extraer métricas filtradas
    print("[4/5] Entrenando modelo y evaluando configuraciones...") 
    # Registrar el reloj del sistema al empezar
    inicio = time.time() 
    # Desatar el ajuste matemático de las combinaciones y pliegues
    grid_search.fit(X_train, y_train) 
    # Registrar el reloj al terminar
    fin = time.time() 
    # Mostrar el tiempo total invertido
    print(f"      -> Búsqueda completada en {round((fin - inicio)/60, 2)} minutos.") 

    # Transformar el resumen del GridSearch a DataFrame para su exportación
    cv_resultados = pd.DataFrame(grid_search.cv_results_)
    # Generar la ruta del archivo de historial CSV
    ruta_cv = os.path.join(dir_resultados, f"Resultados_GridSearch_XGB_{target_name}.csv") 
    # Guardar el reporte de todas las iteraciones en el disco duro
    cv_resultados.to_csv(ruta_cv, index=False) 
    print(f"      -> Evidencia de hiperparámetros guardada en: {ruta_cv}")

    # Filtrar configuraciones que presenten alta estabilidad (baja desviación estándar entre pliegues)
    config_estables = cv_resultados[cv_resultados['std_test_score'] <= 0.10]
    
    if config_estables.empty:
        # Informar problema de estabilidad general
        print("      ADVERTENCIA: Todas las configuraciones tienen std > 0.10.") 
        # Tomar la decisión por defecto de Scikit-Learn si no hay opciones estables
        print("      Se utilizará la de mayor promedio por defecto.") 
        mejor_modelo = grid_search.best_estimator_
    else: 
        # Buscar en qué fila está el F1 promedio más alto de las estables
        mejor_indice = config_estables['mean_test_score'].idxmax() 
        # Sacar el diccionario de hiperparámetros de esa fila ganadora
        mejores_params = config_estables.loc[mejor_indice, 'params'] 
        # Recuperar el valor F1 numérico promedio
        mejor_f1 = config_estables.loc[mejor_indice, 'mean_test_score'] 
        # Recuperar el valor de la desviación estándar
        mejor_std = config_estables.loc[mejor_indice, 'std_test_score'] 
        
        # Anunciar el éxito del hallazgo
        print(f"      -> Mejor configuración estable encontrada:") 
        # Detallar cuáles parámetros ganaron
        print(f"         Hiperparámetros: {mejores_params}") 
        # Mostrar su rendimiento documentado
        print(f"         F1-Macro Promedio: {mejor_f1:.4f} (std: {mejor_std:.4f})") 

        # Crear una copia limpia del estimador base
        mejor_modelo = clone(grid_search.estimator) 
        # Asignarle estrictamente los hiperparámetros que ganaron
        mejor_modelo.set_params(**mejores_params) 
        # Entrenarlo de forma definitiva con el 100% de la matriz de entrenamiento
        mejor_modelo.fit(X_train, y_train) 
        
    # --- GUARDADO DEL MODELO MAESTRO EN DISCO ---
    # Configurar ruta del archivo pickle para el modelo final
    ruta_modelo = os.path.join(dir_resultados, f"Modelo_Optimo_XGBoost_{target_name}.pkl")
    # Exportar el modelo optimizado a disco
    joblib.dump(mejor_modelo, ruta_modelo)
    print(f"      -> Modelo óptimo guardado en: {ruta_modelo}")

    # --- MÉTRICAS DE ENTRENAMIENTO ---
    # Verificar el rendimiento en la misma data de entrenamiento para evaluar posible sobreajuste
    print("\n--- Rendimiento en entrenamiento: ---")
    y_pred_train = mejor_modelo.predict(X_train)
    if is_multiclass:
        print(f"F1-Score (Macro) Train: {f1_score(y_train, y_pred_train, average='macro'):.4f}")
    else:
        print(f"F1-Score (Clase 1) Train: {f1_score(y_train, y_pred_train, pos_label=1):.4f}")

    # Liberar memoria de datos de entrenamiento
    del df_train_maestro, X_train, y_train; gc.collect()

    # [5/5] Evaluación final en Prueba
    print("\n[5/5] Evaluando en conjunto de prueba...")
    # Cargar los datasets de test que no fueron vistos por el modelo
    df_onco_test = pd.read_csv(os.path.join(dir_datos, "dataset_prueba_onco.csv"), low_memory=False)
    df_control_test = pd.read_csv(os.path.join(dir_datos, "dataset_prueba_control.csv"), low_memory=False)

    # Unir ambas tablas y separar features/target
    df_test_maestro = pd.concat([df_onco_test, df_control_test], ignore_index=True)
    X_test = df_test_maestro[features]
    y_test = df_test_maestro[target_name]
    total_instancias = len(y_test)

    # Calcular predicciones duras y probabilidades para el conjunto de evaluación
    y_pred = mejor_modelo.predict(X_test)
    y_pred_proba = mejor_modelo.predict_proba(X_test)

    print("\n--- Resultados finales de evaluación ---") 
    # Mostrar por consola el reporte general de clasificación
    print(classification_report(y_test, y_pred))
    
    # Obtener métricas generales F1-Macro
    f1_macro_val = f1_score(y_test, y_pred, average='macro')
    
    # Evaluar métricas específicas según la naturaleza de la clasificación (Multiclase o Binaria)
    if is_multiclass:
        # Binarizar el conjunto de prueba para el cálculo de métricas One-vs-Rest (OvR)
        y_test_bin = label_binarize(y_test, classes=clases_unicas)
        auc_roc_val = roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='weighted')
        auprc_val = average_precision_score(y_test_bin, y_pred_proba, average='weighted')
        # Calcular el Brier Score promedio para penalizar la incertidumbre de las probabilidades
        brier_val = np.mean([brier_score_loss(y_test_bin[:, k], y_pred_proba[:, k]) for k in range(len(clases_unicas))])
        
        # Calcular tasa base ponderada a partir de las frecuencias de clases
        clases_temp, soportes_clases = np.unique(y_test, return_counts=True)
        tasa_base = sum([(soporte / total_instancias)**2 for soporte in soportes_clases])

        # Imprimir resultados multiclase
        print(f"F1-Score (Macro): {f1_macro_val:.4f}")
        print(f"AUPRC (OvR Weighted): {auprc_val:.4f}")
        print(f"AUC-ROC (OvR Weighted): {auc_roc_val:.4f}")
        print(f"Brier Score (Multiclase): {brier_val:.4f}")
            
    else:
        # Calcular métricas para una clasificación binaria clásica
        f1_clase1_val = f1_score(y_test, y_pred, pos_label=1)
        auc_roc_val = roc_auc_score(y_test, y_pred_proba[:, 1])
        auprc_val = average_precision_score(y_test, y_pred_proba[:, 1])
        brier_val = brier_score_loss(y_test, y_pred_proba[:, 1])
        
        # Extraer la prevalencia (tasa base) específica de la clase positiva (1)
        clases_temp, soportes_clases = np.unique(y_test, return_counts=True)
        indice_clase_1 = np.where(clases_temp == 1)[0][0]
        tasa_base = soportes_clases[indice_clase_1] / total_instancias
        
        # Imprimir resultados binarios
        print(f"F1-Score (Clase 1): {f1_clase1_val:.4f}")
        print(f"F1-Score (Macro): {f1_macro_val:.4f}")
        print(f"AUPRC: {auprc_val:.4f}")
        print(f"AUC-ROC: {auc_roc_val:.4f}")
        print(f"Brier Score: {brier_val:.4f}")

    # --- INICIO BLOQUE DE VALIDACIÓN DE LIFT AUPRC ---
    # Analizar si el modelo mejora significativamente la predicción por puro azar
    print("\n" + "-" * 60)
    print(f"Validación de Lift (en AUPRC): {target_name.upper()}")
    print("-" * 60)
    print(f"Total episodios de prueba: {total_instancias}")
    print(f"Tasa base (Prevalencia Azar): {tasa_base:.4f} ({tasa_base*100:.2f}%)")
    print(f"AUPRC obtenido por el modelo: {auprc_val:.4f}")
    
    # Establecer la métrica de lift comparando el modelo con el azar
    umbral_minimo = tasa_base * 3.0
    lift_real = auprc_val / tasa_base
    
    print(f"Lift real logrado: {lift_real:.2f}x")
    
    # Condición estricta de Lift superior a 3.0 para datasets con prevalencia menor al 15%
    if tasa_base < 0.15: 
        print(f"AUPRC Mínimo exigido (Tasa Base x 3.0): {umbral_minimo:.4f}")
        if auprc_val > umbral_minimo:
            print("Resultado: Cumple condición de Lift > 3.0")
        else:
            print("Resultado: No cumple condición de Lift > 3.0")
    else:
        print("Resultado: Target suficientemente balanceado")
        
    # 7. Extraer Feature Importances (Reemplaza a los Odds Ratios)
    # Extraer del modelo XGBoost el vector que mide la relevancia de cada variable predictora
    importancias = mejor_modelo.feature_importances_ 
    
    # Crear una tabla estructurada cruzando los atributos originales con sus pesos internos
    df_importancias = pd.DataFrame({ 
        'Variable': features, 
        'Importancia_Relativa': importancias 
    }).sort_values(by='Importancia_Relativa', ascending=False) 
    
    # Filtrar y eliminar de la tabla los predictores que fueron completamente ignorados (peso 0)
    df_importancias = df_importancias[df_importancias['Importancia_Relativa'] > 0] 
    
    # Generar la ruta dinámica y exportar la tabla de importancias a un archivo CSV
    ruta_imp = os.path.join(dir_resultados, f"XGB_Importancia_Predictores_{target_name}.csv") 
    df_importancias.to_csv(ruta_imp, index=False) 
    
    # Confirmar guardado exitoso
    print(f"Importancias de variables guardadas en: {ruta_imp}") 
    
    # Borrar variables de evaluación y limpiar la memoria RAM final
    del df_test_maestro, X_test, y_test 
    gc.collect() 
    print("="*60, "\n")

In [8]:
entrenar_evaluar_xgb('MORTALIDAD')

Iniciando entrenamiento y evaluación de XGBOOST para la variable objetivo: MORTALIDAD
[1/5] Cargando datasets de entrenamiento...
[2/5] Generando muestra balanceada...
      -> Dimensiones entrenamiento: (780416, 110) | Clases: 2
[3/5] Configurando Grid Search CV (K=5)...
[4/5] Entrenando modelo y evaluando configuraciones...
Fitting 5 folds for each of 9 candidates, totalling 45 fits
[CV 1/5] END ...learning_rate=0.01, max_depth=3;, score=0.536 total time=   5.8s
[CV 2/5] END ...learning_rate=0.01, max_depth=3;, score=0.537 total time=   4.4s
[CV 3/5] END ...learning_rate=0.01, max_depth=3;, score=0.537 total time=   3.6s
[CV 4/5] END ...learning_rate=0.01, max_depth=3;, score=0.538 total time=   3.7s
[CV 5/5] END ...learning_rate=0.01, max_depth=3;, score=0.536 total time=   3.6s
[CV 1/5] END ...learning_rate=0.01, max_depth=6;, score=0.564 total time=   5.5s
[CV 2/5] END ...learning_rate=0.01, max_depth=6;, score=0.564 total time=   5.3s
[CV 3/5] END ...learning_rate=0.01, max_depth

In [9]:
entrenar_evaluar_xgb('SEVERIDAD')

Iniciando entrenamiento y evaluación de XGBOOST para la variable objetivo: SEVERIDAD
[1/5] Cargando datasets de entrenamiento...
[2/5] Generando muestra balanceada...
      -> Dimensiones entrenamiento: (780416, 110) | Clases: 4
[3/5] Configurando Grid Search CV (K=5)...
[4/5] Entrenando modelo y evaluando configuraciones...
Fitting 5 folds for each of 9 candidates, totalling 45 fits
[CV 1/5] END ...learning_rate=0.01, max_depth=3;, score=0.666 total time=  13.7s
[CV 2/5] END ...learning_rate=0.01, max_depth=3;, score=0.665 total time=  12.1s
[CV 3/5] END ...learning_rate=0.01, max_depth=3;, score=0.666 total time=  12.2s
[CV 4/5] END ...learning_rate=0.01, max_depth=3;, score=0.667 total time=  12.2s
[CV 5/5] END ...learning_rate=0.01, max_depth=3;, score=0.665 total time=  12.5s
[CV 1/5] END ...learning_rate=0.01, max_depth=6;, score=0.706 total time=  18.8s
[CV 2/5] END ...learning_rate=0.01, max_depth=6;, score=0.705 total time=  18.8s
[CV 3/5] END ...learning_rate=0.01, max_depth=

In [10]:
entrenar_evaluar_xgb('CONSUMO_RECURSOS')

Iniciando entrenamiento y evaluación de XGBOOST para la variable objetivo: CONSUMO_RECURSOS
[1/5] Cargando datasets de entrenamiento...
[2/5] Generando muestra balanceada...
      -> Dimensiones entrenamiento: (780416, 110) | Clases: 3
[3/5] Configurando Grid Search CV (K=5)...
[4/5] Entrenando modelo y evaluando configuraciones...
Fitting 5 folds for each of 9 candidates, totalling 45 fits
[CV 1/5] END ...learning_rate=0.01, max_depth=3;, score=0.548 total time=  10.7s
[CV 2/5] END ...learning_rate=0.01, max_depth=3;, score=0.547 total time=   9.3s
[CV 3/5] END ...learning_rate=0.01, max_depth=3;, score=0.545 total time=   9.2s
[CV 4/5] END ...learning_rate=0.01, max_depth=3;, score=0.548 total time=   9.2s
[CV 5/5] END ...learning_rate=0.01, max_depth=3;, score=0.548 total time=  10.8s
[CV 1/5] END ...learning_rate=0.01, max_depth=6;, score=0.631 total time=  13.6s
[CV 2/5] END ...learning_rate=0.01, max_depth=6;, score=0.630 total time=  13.9s
[CV 3/5] END ...learning_rate=0.01, max